# rl-swing — Colab training

Runs PPO/DQN training for the trade-filter env on Colab. Mounts Drive
for artifact persistence so checkpoints survive session timeouts.

**Replace `<USER>`** with the GitHub user/org that owns the repo, or
switch to your fork. Public repo: no auth needed. Private repo: see
the optional PAT cell.


In [ ]:
# 1) Clone the repo (public).
GITHUB_USER = "<USER>"   # <-- edit me
REPO = "trading-bot-rl"
BRANCH = "main"

!git clone --depth=1 -b {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}


In [ ]:
# 1b) Private repo? Use a PAT instead. Skip this cell if your repo is public.
# from getpass import getpass
# pat = getpass('GitHub PAT: ')
# !git clone --depth=1 -b {BRANCH} https://{pat}@github.com/{GITHUB_USER}/{REPO}.git
# %cd {REPO}


In [ ]:
# 2) Install the package and its dependencies.
!pip install -q -e .
import rl_swing; print('rl_swing version', rl_swing.__version__)


In [ ]:
# 3) Mount Drive for artifact persistence (recommended).
from google.colab import drive
drive.mount('/content/drive')
ARTIFACT_ROOT = '/content/drive/MyDrive/rl-swing/models'
import os; os.makedirs(ARTIFACT_ROOT, exist_ok=True)
ARTIFACT_ROOT


In [ ]:
# 4) Smoke run — verify the env + training loop work in Colab.
from rl_swing.rl.training.colab_entrypoint import train
summary = train(
    experiment='configs/experiments/ppo_filter_smoke.yaml',
    total_timesteps=10_000,
    seed=11,
    artifact_root=ARTIFACT_ROOT,
)
summary['runs'][0]['best_validation_score'], summary['runs'][0]['eval_history']


In [ ]:
# 5) Real run — 500k steps, 3 seeds. (Estimate: 1-3 hours on a CPU runtime.)
summary = train(
    experiment='configs/experiments/ppo_filter_v001.yaml',
    total_timesteps=500_000,
    seeds=[11, 22, 33],
    artifact_root=ARTIFACT_ROOT,
)
[r['best_validation_score'] for r in summary['runs']]


## Reading the artifacts

Each `train(...)` call writes:

* `<artifact_root>/<experiment>/seed_<N>/best.zip`
* `<artifact_root>/<experiment>/seed_<N>/last.zip`
* `<artifact_root>/<experiment>/seed_<N>/metadata.json`
* `<artifact_root>/<experiment>/model.zip` (alias for the best across seeds)
* `<artifact_root>/<experiment>/training_summary.json`

Run `rl-swing validate --experiment ...` locally against the same
experiment YAML to produce the walk-forward comparison report.
